# 🌊 CalCOFI Marine Data Analysis
### California Cooperative Oceanic Fisheries Investigations
---
This notebook fetches, cleans, and visualizes two CalCOFI datasets from NOAA ERDDAP:
- **Zooplankton Volume Data** (`erdCalCOFIzoovol`) — biological/ecosystem features
- **Hydrographic Bottle Data** (`erdCalCOFIBottle`) — water chemistry features

> **Note:** If live ERDDAP queries are slow or time out, synthetic demo data is generated automatically so you can still explore all visualizations.

## 📦 1. Install & Import Libraries

In [ ]:
# Install required libraries if needed
# !pip install pandas requests matplotlib seaborn plotly folium scipy --quiet

import pandas as pd
import numpy as np
import requests
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from io import StringIO

warnings.filterwarnings('ignore')

# ── Notebook styling ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1b2a',
    'axes.facecolor':   '#0d1b2a',
    'axes.edgecolor':   '#1e3a5f',
    'axes.labelcolor':  '#cfe8ff',
    'xtick.color':      '#cfe8ff',
    'ytick.color':      '#cfe8ff',
    'text.color':       '#cfe8ff',
    'grid.color':       '#1e3a5f',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   14,
    'axes.labelsize':   11,
})
OCEAN_PALETTE = ['#00b4d8', '#0077b6', '#48cae4', '#90e0ef', '#ade8f4', '#caf0f8']
print('✅ Libraries loaded successfully')

## 🌐 2. Fetch Data from NOAA ERDDAP

In [ ]:
BASE_URL = "https://oceanview.pfeg.noaa.gov/erddap/tabledap/"

def fetch_erddap_data(dataset_id, variables, time_filter=None, max_rows=5000):
    """
    Fetch tabular data from NOAA ERDDAP.
    Returns a DataFrame or None on failure.
    """
    var_str = ','.join(variables)
    url = f"{BASE_URL}{dataset_id}.csv?{var_str}"
    if time_filter:
        url += f"&{time_filter}"
    url += f"&orderByLimit(\"{max_rows}\")"

    print(f"  Fetching: {url[:90]}...")
    try:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        # ERDDAP CSV has 2 header rows: column names + units → skip units row
        df = pd.read_csv(StringIO(resp.text), skiprows=[1])
        print(f"  ✅ {dataset_id}: {df.shape[0]:,} rows × {df.shape[1]} columns")
        return df
    except Exception as e:
        print(f"  ⚠️  {dataset_id} fetch failed: {e}")
        return None

# ── Zooplankton ───────────────────────────────────────────────────────────────
zoo_vars = ['time','latitude','longitude','line','station',
            'volume_sampled','small_plankton','total_plankton']
print("Fetching Zooplankton data …")
zoo_df = fetch_erddap_data(
    "erdCalCOFIzoovol", zoo_vars,
    time_filter="time>=2018-01-01T00:00:00Z"
)

# ── Bottle (hydrographic) ─────────────────────────────────────────────────────
bottle_vars = ['Depthm','T_degC','Salnty','Oxy_µmol/Kg','PO4uM',
               'SiO3uM','NO2uM','NO3uM','NH3uM','ChlorA',
               'DIC1','TA1','pH1','Lat_Dec','Lon_Dec','Date','Time']
print("\nFetching Bottle (hydrographic) data …")
bottle_df = fetch_erddap_data(
    "erdCalCOFIBottle", bottle_vars,
    time_filter="Date>=20180101"
)

## 🔧 3. Synthetic Fallback (runs only if ERDDAP is unavailable)

In [ ]:
rng = np.random.default_rng(42)

def make_synthetic_zoo(n=1200):
    dates = pd.date_range('2018-01-01', '2024-12-31', periods=n)
    lat   = rng.uniform(30, 37, n)
    lon   = rng.uniform(-122, -117, n)
    total = np.abs(rng.normal(350, 120, n))
    return pd.DataFrame({
        'time':           dates,
        'latitude':       lat,
        'longitude':      lon,
        'line':           rng.choice([76.7, 80.0, 83.3, 86.7, 90.0, 93.3], n),
        'station':        rng.integers(28, 120, n),
        'volume_sampled': np.abs(rng.normal(50, 15, n)),
        'small_plankton': total * rng.uniform(0.2, 0.6, n),
        'total_plankton': total,
    })

def make_synthetic_bottle(n=3000):
    depth = rng.uniform(0, 500, n)
    temp  = 18 - depth * 0.028 + rng.normal(0, 1.5, n)
    sal   = 33.5 + depth * 0.003 + rng.normal(0, 0.2, n)
    oxy   = 280 - depth * 0.35 + rng.normal(0, 20, n)
    return pd.DataFrame({
        'Depthm':      depth,
        'T_degC':      temp,
        'Salnty':      sal,
        'Oxy_µmol/Kg': oxy,
        'PO4uM':       rng.uniform(0.05, 3.5, n),
        'SiO3uM':      rng.uniform(1, 80, n),
        'NO2uM':       rng.uniform(0, 0.8, n),
        'NO3uM':       rng.uniform(0.1, 40, n),
        'NH3uM':       rng.uniform(0, 1.5, n),
        'ChlorA':      np.abs(rng.exponential(0.6, n)),
        'DIC1':        rng.uniform(1900, 2300, n),
        'TA1':         rng.uniform(2150, 2450, n),
        'pH1':         rng.uniform(7.6, 8.3, n),
        'Lat_Dec':     rng.uniform(30, 37, n),
        'Lon_Dec':     rng.uniform(-122, -117, n),
        'Date':        pd.date_range('2018-01-01', '2024-12-31', periods=n),
        'Time':        ['12:00:00'] * n,
    })

if zoo_df is None:
    print("Using synthetic zooplankton data")
    zoo_df = make_synthetic_zoo()
if bottle_df is None:
    print("Using synthetic bottle data")
    bottle_df = make_synthetic_bottle()

print(f"\nZooplankton shape : {zoo_df.shape}")
print(f"Bottle shape      : {bottle_df.shape}")

## 🧹 4. Data Cleaning

In [ ]:
# ── Zooplankton cleaning ──────────────────────────────────────────────────────
zoo = zoo_df.copy()
zoo['time'] = pd.to_datetime(zoo['time'], errors='coerce')
zoo = zoo.dropna(subset=['time','latitude','longitude','total_plankton'])
zoo = zoo[zoo['total_plankton'] > 0]
zoo = zoo[zoo['volume_sampled'] > 0]
zoo['year']  = zoo['time'].dt.year
zoo['month'] = zoo['time'].dt.month
zoo['plankton_density'] = zoo['total_plankton'] / zoo['volume_sampled']
zoo['small_fraction']   = zoo['small_plankton'] / zoo['total_plankton']

print("── Zooplankton after cleaning ──")
print(f"  Rows : {len(zoo):,}")
print(f"  Years: {zoo['year'].min()} – {zoo['year'].max()}")
print(f"  NaN %: {zoo.isnull().mean().mul(100).round(1).to_dict()}\n")

# ── Bottle cleaning ───────────────────────────────────────────────────────────
bot = bottle_df.copy()
bot.rename(columns={'Oxy_µmol/Kg': 'Oxygen'}, inplace=True)
bot['Date'] = pd.to_datetime(bot['Date'].astype(str), errors='coerce')
numeric_cols = ['Depthm','T_degC','Salnty','Oxygen','PO4uM','SiO3uM',
                'NO2uM','NO3uM','NH3uM','ChlorA','DIC1','TA1','pH1']
for c in numeric_cols:
    bot[c] = pd.to_numeric(bot[c], errors='coerce')

# Remove physically impossible values
bot = bot[(bot['T_degC'].between(-2, 35)) | bot['T_degC'].isna()]
bot = bot[(bot['Salnty'].between(28, 38)) | bot['Salnty'].isna()]
bot = bot[(bot['Oxygen'] >= 0) | bot['Oxygen'].isna()]
bot = bot[(bot['pH1'].between(7.0, 8.6)) | bot['pH1'].isna()]
bot = bot.dropna(subset=['Depthm','T_degC','Salnty'])
bot['year']  = bot['Date'].dt.year
bot['month'] = bot['Date'].dt.month

print("── Bottle after cleaning ──")
print(f"  Rows : {len(bot):,}")
print(f"  Depth range: {bot['Depthm'].min():.0f} – {bot['Depthm'].max():.0f} m")
print(bot[numeric_cols].describe().round(2))

## 📊 5. Zooplankton Visualizations

In [ ]:
# ── 5a. Plankton density over time (monthly mean) ─────────────────────────────
monthly = zoo.groupby(zoo['time'].dt.to_period('M'))['plankton_density'].mean().reset_index()
monthly['time'] = monthly['time'].dt.to_timestamp()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor('#0d1b2a')

ax = axes[0]
ax.plot(monthly['time'], monthly['plankton_density'],
        color='#00b4d8', linewidth=2, alpha=0.9)
ax.fill_between(monthly['time'], monthly['plankton_density'],
                alpha=0.15, color='#00b4d8')
ax.set_title('Monthly Mean Plankton Density', fontweight='bold', color='#cfe8ff')
ax.set_ylabel('Plankton Density (mL/m³)', color='#cfe8ff')
ax.grid(True)

# ── 5b. Annual boxplot ────────────────────────────────────────────────────────
ax2 = axes[1]
years_sorted = sorted(zoo['year'].unique())
data_by_year = [zoo[zoo['year']==y]['plankton_density'].dropna().values for y in years_sorted]
bp = ax2.boxplot(data_by_year, labels=years_sorted,
                 patch_artist=True, notch=True,
                 medianprops=dict(color='#ade8f4', linewidth=2),
                 whiskerprops=dict(color='#48cae4'),
                 capprops=dict(color='#48cae4'),
                 flierprops=dict(marker='o', color='#90e0ef', alpha=0.4, markersize=3))
for patch, color in zip(bp['boxes'], sns.color_palette("Blues_d", len(years_sorted))):
    patch.set_facecolor((*color[:3], 0.5))
ax2.set_title('Plankton Density by Year', fontweight='bold', color='#cfe8ff')
ax2.set_ylabel('Plankton Density (mL/m³)', color='#cfe8ff')
ax2.set_xlabel('Year', color='#cfe8ff')
ax2.grid(True, axis='y')

plt.tight_layout()
plt.suptitle('Zooplankton Temporal Patterns — CalCOFI',
             y=1.02, fontsize=15, fontweight='bold', color='#cfe8ff')
plt.show()

In [ ]:
# ── 5c. Spatial scatter map of zooplankton ────────────────────────────────────
fig = px.scatter_geo(
    zoo, lat='latitude', lon='longitude',
    color='plankton_density',
    size='total_plankton',
    hover_data={'latitude':True,'longitude':True,
                'plankton_density':':.2f','year':True},
    color_continuous_scale='Blues',
    title='🌊 Zooplankton Density — Spatial Distribution',
    size_max=18,
    projection='natural earth',
    scope='north america',
    center=dict(lat=34, lon=-120),
    fitbounds='locations',
    basemap_visible=True,
)
fig.update_layout(
    paper_bgcolor='#0d1b2a', font_color='#cfe8ff',
    title_font_size=16, coloraxis_colorbar_title='Density'
)
fig.show()

In [ ]:
# ── 5d. Seasonal heatmap ──────────────────────────────────────────────────────
seasonal = zoo.groupby(['year','month'])['plankton_density'].mean().unstack()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
seasonal.columns = [month_names[m-1] for m in seasonal.columns]

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#0d1b2a')
sns.heatmap(seasonal, cmap='YlGnBu', ax=ax, linewidths=0.4,
            annot=True, fmt='.0f', annot_kws={'size': 8},
            cbar_kws={'label': 'Plankton Density'})
ax.set_title('Seasonal Plankton Density Heatmap (Year × Month)',
             fontweight='bold', color='#cfe8ff', pad=12)
ax.set_xlabel('Month', color='#cfe8ff')
ax.set_ylabel('Year', color='#cfe8ff')
plt.tight_layout()
plt.show()

## 🧪 6. Hydrographic Bottle Visualizations

In [ ]:
# ── 6a. Temperature & Salinity depth profiles ─────────────────────────────────
depth_bins = pd.cut(bot['Depthm'], bins=np.arange(0, 520, 20))
depth_profile = bot.groupby(depth_bins, observed=True)[['T_degC','Salnty','Oxygen']].mean()
depth_profile.index = [f"{int(i.left)}" for i in depth_profile.index]

fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharey=False)
fig.patch.set_facecolor('#0d1b2a')

labels = ['Temperature (°C)', 'Salinity (psu)', 'Oxygen (µmol/kg)']
colors = ['#ff6b6b', '#48cae4', '#90e0ef']
cols   = ['T_degC', 'Salnty', 'Oxygen']

for ax, col, label, color in zip(axes, cols, labels, colors):
    vals  = depth_profile[col].values
    depths = np.array([int(i) for i in depth_profile.index])
    ax.plot(vals, depths, color=color, linewidth=2.5)
    ax.fill_betweenx(depths, vals, alpha=0.1, color=color)
    ax.scatter(vals, depths, s=18, color=color, zorder=5)
    ax.invert_yaxis()
    ax.set_xlabel(label, color='#cfe8ff')
    ax.set_ylabel('Depth (m)', color='#cfe8ff')
    ax.set_title(label, fontweight='bold', color='#cfe8ff')
    ax.grid(True)

plt.suptitle('Mean Depth Profiles — CalCOFI Bottle Data',
             fontsize=15, fontweight='bold', color='#cfe8ff', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6b. T-S Diagram (coloured by depth) ─────────────────────────────────────
fig = px.scatter(
    bot.sample(min(2000, len(bot)), random_state=42),
    x='Salnty', y='T_degC', color='Depthm',
    color_continuous_scale='Blues_r',
    opacity=0.6,
    labels={'Salnty':'Salinity (psu)','T_degC':'Temperature (°C)','Depthm':'Depth (m)'},
    title='🌡️ T-S Diagram — Water Mass Properties',
    hover_data=['pH1','Oxygen']
)
fig.update_layout(
    paper_bgcolor='#0d1b2a', plot_bgcolor='#0d1b2a',
    font_color='#cfe8ff', title_font_size=15
)
fig.show()

In [ ]:
# ── 6c. Nutrient correlation heatmap ──────────────────────────────────────────
nutrient_cols = ['T_degC','Salnty','Oxygen','PO4uM','SiO3uM',
                 'NO3uM','ChlorA','DIC1','pH1']
corr = bot[nutrient_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0d1b2a')
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Pearson r'})
ax.set_title('Nutrient & Hydrographic Variable Correlations',
             fontweight='bold', color='#cfe8ff', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6d. Chlorophyll-A vs. nutrients (faceted scatter) ─────────────────────────
bot_surf = bot[bot['Depthm'] < 50].dropna(subset=['ChlorA','NO3uM','PO4uM','SiO3uM'])

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Chl-A vs NO₃', 'Chl-A vs PO₄', 'Chl-A vs SiO₃']
)
for i, (xcol, label) in enumerate([('NO3uM','NO₃ (µM)'),
                                     ('PO4uM','PO₄ (µM)'),
                                     ('SiO3uM','SiO₃ (µM)')], start=1):
    fig.add_trace(
        go.Scatter(x=bot_surf[xcol], y=bot_surf['ChlorA'],
                   mode='markers',
                   marker=dict(color='#48cae4', opacity=0.4, size=5),
                   name=label),
        row=1, col=i
    )

fig.update_layout(
    title_text='🌿 Surface Chlorophyll-A vs. Nutrients (Depth < 50 m)',
    paper_bgcolor='#0d1b2a', plot_bgcolor='#0d1b2a',
    font_color='#cfe8ff', showlegend=False, height=420
)
fig.update_yaxes(title_text='Chlorophyll-A (µg/L)')
fig.show()

In [ ]:
# ── 6e. Spatial map of pH at surface ─────────────────────────────────────────
bot_map = bot[(bot['Depthm'] < 30) & bot['pH1'].notna() &
              bot['Lat_Dec'].notna() & bot['Lon_Dec'].notna()]

fig = px.scatter_geo(
    bot_map.sample(min(1500, len(bot_map)), random_state=0),
    lat='Lat_Dec', lon='Lon_Dec',
    color='pH1',
    color_continuous_scale='RdYlBu',
    range_color=[7.7, 8.3],
    size_max=10,
    hover_data=['T_degC','Salnty','Oxygen'],
    title='🧪 Surface Ocean pH — CalCOFI Region',
    fitbounds='locations',
    scope='north america',
)
fig.update_layout(
    paper_bgcolor='#0d1b2a', font_color='#cfe8ff',
    coloraxis_colorbar_title='pH'
)
fig.show()

## 🔗 7. Merged Dataset Overview

In [ ]:
# Grid-cell merge: round lat/lon to 0.5° and match
zoo_merge = zoo.copy()
bot_merge = bot.copy()

zoo_merge['lat_r'] = zoo_merge['latitude'].round(1)
zoo_merge['lon_r'] = zoo_merge['longitude'].round(1)

bot_merge['lat_r'] = bot_merge['Lat_Dec'].round(1)
bot_merge['lon_r'] = bot_merge['Lon_Dec'].round(1)

zoo_agg = zoo_merge.groupby(['lat_r','lon_r'])['plankton_density'].mean().reset_index()
bot_agg = bot_merge.groupby(['lat_r','lon_r'])[['T_degC','Salnty','ChlorA','NO3uM','pH1']].mean().reset_index()

merged = pd.merge(zoo_agg, bot_agg, on=['lat_r','lon_r'], how='inner')
print(f"Merged grid cells: {len(merged)}")

# Correlation of zooplankton density with environmental variables
env_vars = ['T_degC','Salnty','ChlorA','NO3uM','pH1']
corr_vec = merged[env_vars].corrwith(merged['plankton_density']).round(3).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('#0d1b2a')
colors_bar = ['#ff6b6b' if v < 0 else '#48cae4' for v in corr_vec]
corr_vec.plot(kind='barh', ax=ax, color=colors_bar, edgecolor='#0d1b2a')
ax.axvline(0, color='#cfe8ff', linewidth=0.8)
ax.set_title('Zooplankton Density vs. Environmental Variables (Pearson r)',
             fontweight='bold', color='#cfe8ff')
ax.set_xlabel('Pearson r', color='#cfe8ff')
ax.grid(True, axis='x')
plt.tight_layout()
plt.show()

## 📝 8. Summary Statistics

In [ ]:
print("═" * 55)
print("  ZOOPLANKTON SUMMARY")
print("═" * 55)
print(zoo[['plankton_density','total_plankton','small_fraction']].describe().round(3).to_string())

print("\n" + "═" * 55)
print("  BOTTLE (HYDROGRAPHIC) SUMMARY")
print("═" * 55)
print(bot[['T_degC','Salnty','Oxygen','ChlorA','NO3uM','pH1']].describe().round(3).to_string())

print("\n✅ Analysis complete.")